# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL:https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport json# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadata# Note: dataset.metadata is a CroissantEntity, access fields as attributesprint(f"Dataset Name: {metadata.name}")print(f"Description: {metadata.description}\n")

## 2. Data OverviewReview available record sets, fields, columns, and their `@id`s.Let's enumerate all record sets in the dataset, and for each record set, list its fields and columns referencing their `@id`s.

In [ ]:
# Review record sets and their field/column idsprint("Record sets in this dataset:")record_sets = list(dataset.record_sets)record_set_ids = []for rs in record_sets:    print(f"- Record Set @id: {rs['@id']}")    record_set_ids.append(rs['@id'])    # List fields    fields = rs.get('field', [])    if isinstance(fields, dict):        fields = [fields]    print("  Fields (by @id):")    for f in fields:        if isinstance(f, dict):            print(f"    - {f.get('@id', str(f))}")        else:            print(f"    - {f}")    # List columns if present    columns = rs.get('column', [])    if isinstance(columns, dict):        columns = [columns]    if columns:        print("  Columns (by @id):")        for c in columns:            if isinstance(c, dict):                print(f"    - {c.get('@id', str(c))}")            else:                print(f"    - {c}")print("\nIf no record sets are shown, check the Croissant schema for available record sets.")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis.Below we load records from all available record sets (by `@id`). Adjust the `record_set_ids` list if you wish to focus on specific record sets.

In [ ]:
# Extract data from each record setdataframes = {}for record_set_id in record_set_ids:    print(f"Loading records for record set: {record_set_id}")    try:        records = list(dataset.records(record_set=record_set_id))        if records:            df = pd.DataFrame(records)            dataframes[record_set_id] = df            print(f"Columns: {df.columns.tolist()}")            print(df.head(2))        else:            print(f"No records found for {record_set_id}.")    except Exception as e:        print(f"Could not load record set {record_set_id}: {e}")print("\nLoaded DataFrames for record sets:", list(dataframes.keys()))

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.For this section, pick a DataFrame with numeric fields (adjust as needed for your dataset). You might need to inspect the field names and data types first.

In [ ]:
# Choose the primary record set for analysisif dataframes:    example_rs_id = list(dataframes.keys())[0]  # Use the first available record set    df = dataframes[example_rs_id]    print(f"Working with record set: {example_rs_id}")    # Show numeric columns    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()    print(f"Numeric fields: {numeric_cols}")    if numeric_cols:        numeric_field = numeric_cols[0]  # Use the first numeric column available        print(f"Analyzing numeric field: {numeric_field}")        threshold = df[numeric_field].mean()  # Example threshold: mean        # Filtering        filtered_df = df[df[numeric_field] > threshold].copy()        print(f"Filtered records where {numeric_field} > {threshold:.3f}:")        print(filtered_df.head())        # Normalization        norm_col = f"{numeric_field}_normalized"        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()        print(filtered_df[[numeric_field, norm_col]].head())        # Try grouping by a categorical column        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()        group_field = None        for col in cat_cols:            # Try grouping by the first categorical field with more than one value            if df[col].nunique() > 1:                group_field = col                break        if group_field:            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()            print(f"\nGrouped mean of {numeric_field} by {group_field}:")            print(grouped_df.head())        else:            print("No suitable categorical group field found.")    else:        print("No numeric fields found for EDA in this record set.")else:    print("No dataframes loaded to analyze.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.For demonstration, we'll plot the distribution of the selected numeric field and compare groups if applicable.

In [ ]:
import matplotlib.pyplot as pltif dataframes:    if numeric_cols:        plt.figure(figsize=(8, 5))        df[numeric_field].hist(bins=30, color='skyblue', edgecolor='black')        plt.xlabel(numeric_field)        plt.ylabel('Count')        plt.title(f'Distribution of {numeric_field}')        plt.show()        if group_field:            plt.figure(figsize=(8, 5))            df.groupby(group_field)[numeric_field].mean().plot(kind='bar')            plt.xlabel(group_field)            plt.ylabel(f'Mean {numeric_field}')            plt.title(f'Mean {numeric_field} by {group_field}')            plt.show()else:    print("No visualization possible: no numeric fields or DataFrames loaded.")

## 6. ConclusionIn this notebook, we demonstrated how to load, explore, and analyze a Croissant-formatted biomolecular dataset using the `mlcroissant` library.- We loaded and inspected available record sets and their fields using `@id` references.- Extracted data into pandas DataFrames and performed basic exploratory analysis.- Applied common EDA techniques: filtering, normalization, and grouping.- Visualized distributions and relationships in the dataset.Use the notebook as a baseline to adapt detailed analyses for your specific research questions leveraging the FAIR² dataset.